# Entrenamiento en Google Colab (GPU T4)

> ⚠️ **NO ES UNA HERRAMIENTA DIAGNÓSTICA.** Proyecto educativo y experimental.

Flujo completo: datos → manifiesto con split por paciente → fine-tuning →
evaluación → Grad-CAM → validación externa → análisis de sesgo.

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno → GPU`.

## 1. Repositorio y dependencias

In [ ]:
!git clone https://github.com/GGGuardin/chest-xray-pneumonia.git repo
%cd repo
!pip install -q albumentations pydicom timm

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO DISPONIBLE')

## 2. Descarga del dataset

Sube tu `kaggle.json` (Kaggle → Settings → API → Create New Token).
Hay que aceptar las reglas de la competición RSNA en la web antes de descargar.

In [ ]:
from google.colab import files
files.upload()  # kaggle.json

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle competitions download -c rsna-pneumonia-detection-challenge -p data/raw/rsna
!unzip -q data/raw/rsna/rsna-pneumonia-detection-challenge.zip -d data/raw/rsna
!ls data/raw/rsna | head

## 3. Manifiesto y split POR PACIENTE

Lee además sexo, edad y proyección AP/PA de las cabeceras DICOM (tarda unos
minutos sobre 26k ficheros), que es lo que habilita el análisis de subgrupos.

In [ ]:
!python -m src.prepare_data --dataset rsna --root data/raw/rsna --out data/manifest_rsna.csv

## 4. Exploración rápida

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from src.data import load_image, split_summary

df = pd.read_csv('data/manifest_rsna.csv')
print(split_summary(df).to_string())
print('\nProyección:\n', df['view'].value_counts())
print('\nSexo:\n', df['sex'].value_counts())

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (_, r) in zip(axes[0], df[df.label == 0].sample(4, random_state=0).iterrows()):
    ax.imshow(load_image(r.image_path), cmap='gray'); ax.set_title('NORMAL'); ax.axis('off')
for ax, (_, r) in zip(axes[1], df[df.label == 1].sample(4, random_state=0).iterrows()):
    ax.imshow(load_image(r.image_path), cmap='gray'); ax.set_title('OPACIDAD'); ax.axis('off')
plt.tight_layout()

## 5. Entrenamiento

En una T4, ~12 épocas de DenseNet-121 a 224×224 sobre RSNA rondan 1,5-2 h.
El checkpoint se guarda en cada mejora de AUROC de validación, así que una
desconexión de Colab no pierde el trabajo.

In [ ]:
!python -m src.train --config configs/rsna.yaml

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
h = pd.read_csv('runs/rsna_densenet121/history.csv')
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h.epoch, h.train_loss, label='train'); ax[0].plot(h.epoch, h.val_loss, label='val')
ax[0].set_title('loss'); ax[0].set_xlabel('época'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(h.epoch, h.train_auroc, label='train'); ax[1].plot(h.epoch, h.val_auroc, label='val')
ax[1].set_title('AUROC'); ax[1].set_xlabel('época'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout()

## 6. Evaluación en test (split por paciente)

In [ ]:
!python -m src.evaluate --checkpoint runs/rsna_densenet121/best.pth \
    --manifest data/manifest_rsna.csv --split test --out-dir reports/rsna_test

## 7. Grad-CAM y auditoría de atajos

Si la energía del mapa cae en bordes, esquinas o marcadores en vez de en el
pulmón, la predicción se apoya en un atajo espurio.

In [ ]:
!python -m src.explain --checkpoint runs/rsna_densenet121/best.pth \
    --manifest data/manifest_rsna.csv --split test --n 12 --out-dir reports/gradcam

from IPython.display import Image, display
import glob
for p in sorted(glob.glob('reports/gradcam/*.png'))[:6]:
    display(Image(p))

## 8. Validación externa

Criterio del proyecto: una caída de AUROC > 0,10 respecto al test interno es un
problema de generalización que hay que documentar y analizar.

In [ ]:
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p data/raw
!unzip -q data/raw/chest-xray-pneumonia.zip -d data/raw
!python -m src.prepare_data --dataset kaggle_pneumonia --root data/raw/chest_xray \
    --out data/manifest_kaggle.csv
!python -m src.evaluate --checkpoint runs/rsna_densenet121/best.pth \
    --manifest data/manifest_kaggle.csv --split all --out-dir reports/externo_kaggle

## 9. Análisis de sesgo por subgrupos

In [ ]:
!python -m src.fairness --predictions reports/rsna_test/predictions.csv \
    --out-dir reports/fairness

from IPython.display import Image, display
display(Image('reports/fairness/fnr_por_subgrupo.png'))

## 10. Descargar el checkpoint para la demo

Con este `best.pth` se despliega la demo Gradio en Hugging Face Spaces.

In [ ]:
from google.colab import files
files.download('runs/rsna_densenet121/best.pth')